# SAE Feature $\leftrightarrow$ Neuron Mechanistic Analysis for Bias Detection

**Attention Atlas — Master's Thesis Mechanistic Interpretability Study**

This notebook investigates the **relationship between SAE latent features and individual
neurons** in GPT-2, with the goal of mechanistically understanding how social
bias is encoded in transformer internals.

### Research Questions

1. **Decoder anatomy**: What is the neuronal composition of bias-discriminative SAE features?
2. **Polysemanticity**: How polysemantic are the neurons involved in bias encoding?
3. **Superposition**: To what extent is bias information stored in superposition?
4. **MLP circuits**: Which MLP neurons are causally responsible for bias features?
5. **Cross-layer tracking**: Do the same neurons carry bias signal across layers?
6. **Triangulation**: How do SAE-identified neurons relate to attention-head importance?

### Theoretical Background

The **superposition hypothesis** (Elhage et al., 2022) posits that neural networks represent
more features than they have dimensions by encoding features as nearly-orthogonal directions
in activation space. **Sparse Autoencoders** (Cunningham et al., 2023; Bricken et al., 2023)
recover these features by learning an overcomplete dictionary under a sparsity constraint.

The key insight for this analysis is that the **decoder weight matrix** $W_{\text{dec}}$
provides a direct mapping from SAE features back to neurons:

$$\hat{x} = W_{\text{dec}} z + b_{\text{dec}}$$

where each column $W_{\text{dec}}[:, f]$ is the **decoder direction** of feature $f$ —
a 768-dimensional vector telling us exactly which neurons that feature reconstructs.

> **Runtime:** GPU recommended (T4 or better). CPU fallback supported.

### References

- Elhage et al., *Toy Models of Superposition* (Anthropic, 2022)
- Cunningham et al., *Sparse Autoencoders Find Highly Interpretable Features in Language Models* (2023)
- Bricken et al., *Towards Monosemanticity* (Anthropic, 2023)
- Conmy et al., *Towards Automated Circuit Discovery for Mechanistic Interpretability* (2023)
- Bills et al., *Language models can explain neurons in language models* (OpenAI, 2023)

In [ ]:
!pip install -q transformer-lens einops jaxtyping transformers torch umap-learn scikit-learn scipy seaborn matplotlib pandas
# Show which numpy we got — transformer-lens may have downgraded it
import numpy as np
print(f"numpy version after install: {np.__version__}")

In [ ]:
# IMPORTANT: Restart runtime to clear cached numpy binaries.
# After this cell, the runtime will restart. Then run all cells from the imports cell onward.
import IPython
IPython.Application.instance().kernel.do_shutdown(True)

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset

import numpy as np
import pandas as pd
import json
import gc
from pathlib import Path
from collections import defaultdict
from tqdm.auto import tqdm

import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from scipy import stats
from scipy.cluster.hierarchy import dendrogram, linkage, fcluster
from scipy.spatial.distance import squareform, pdist, cosine
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, average_precision_score, f1_score
from sklearn.model_selection import train_test_split
from sklearn.manifold import TSNE

try:
    import umap
    HAS_UMAP = True
except ImportError:
    HAS_UMAP = False
    print("UMAP not available, will use t-SNE instead")

from transformer_lens import HookedTransformer

sns.set_theme(style="whitegrid", font_scale=1.1)
plt.rcParams.update({
    "figure.dpi": 120,
    "savefig.dpi": 300,
    "savefig.bbox": "tight",
    "font.family": "serif",
})

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")
print(f"PyTorch: {torch.__version__}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

## 1. Configuration

In [ ]:
# ── Model Configuration ─────────────────────────────────────────
# CHANGE THIS to switch between models. Run the full notebook once per model.
# Results are saved to MODEL_OUT; the final section compares both.

MODEL_NAME      = "gpt2"               # OPTIONS: "gpt2", "bert-base-uncased"
MODEL_FAMILY    = "gpt2"               # OPTIONS: "gpt2", "bert"
MODEL_LABEL     = "GPT-2"             # OPTIONS: "GPT-2", "BERT"

# ── Uncomment below for BERT instead ──
# MODEL_NAME      = "bert-base-uncased"
# MODEL_FAMILY    = "bert"
# MODEL_LABEL     = "BERT"

N_LAYERS        = 12               # Number of transformer layers (same for both)
D_MODEL         = 768              # Residual stream dimension (same for both)
D_MLP           = 3072             # MLP hidden dimension (same for both)
N_HEADS         = 12               # Number of attention heads (same for both)
D_HEAD          = 64               # Head dimension (same for both)

# ── SAE Architecture ───────────────────────────────────────────
EXPANSION       = 8                # Dictionary expansion factor
D_SAE           = D_MODEL * EXPANSION  # 6144 latent features
L1_COEFF        = 5e-4             # Sparsity penalty
SAE_LR          = 3e-4             # Learning rate
SAE_EPOCHS      = 75               # Training epochs
SAE_BATCH_SIZE  = 128              # Batch size
VAL_FRACTION    = 0.15             # Validation split

# ── Analysis Configuration ─────────────────────────────────────
ANALYSIS_LAYERS = list(range(12))  # Run full analysis on ALL layers
TOP_K_FEATURES  = 50               # Number of top bias features to analyze
TOP_K_NEURONS   = 30               # Number of top neurons to track
N_ABLATE_NEURONS = 40              # Neurons to ablate (top SAE + control)
ABLATION_SAMPLES = 500             # Samples for causal ablation

# ── Reproducibility ────────────────────────────────────────────
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

# ── Output directories ─────────────────────────────────────────
OUT_DIR = Path("sae_neuron_analysis_results")
OUT_DIR.mkdir(exist_ok=True)
MODEL_OUT = OUT_DIR / MODEL_LABEL.lower().replace(" ", "_")
MODEL_OUT.mkdir(exist_ok=True)

print(f"Model: {MODEL_LABEL} ({MODEL_NAME})")
print(f"SAE: {D_MODEL} -> {D_SAE} (x{EXPANSION} expansion)")
print(f"Analysis layers: ALL ({len(ANALYSIS_LAYERS)} layers)")
print(f"Ablation: {N_ABLATE_NEURONS} neurons x {ABLATION_SAMPLES} samples per layer")
print(f"Output: {MODEL_OUT}")

## 2. Dataset Loading

We load the bias dataset and prepare it for activation extraction.
The dataset contains sentences labeled as biased or neutral, with bias category annotations.

In [ ]:
# ── Mount Google Drive and load dataset ──
from google.colab import drive
drive.mount("/content/drive")

DRIVE_DATASET_DIR = Path("/content/drive/MyDrive/dataset/bias")
DATASET_PATH = str(DRIVE_DATASET_DIR / "bias_sentences_v6.json")

if not Path(DATASET_PATH).exists():
    print(f"File not found: {DATASET_PATH}")
    print(f"\nFiles in {DRIVE_DATASET_DIR}:")
    if DRIVE_DATASET_DIR.exists():
        for f in sorted(DRIVE_DATASET_DIR.iterdir()):
            print(f"  {f.name}")
    else:
        print("  Directory not found! Check your Drive path.")
    raise FileNotFoundError("bias_sentences_v6.json not found. Adjust path above.")

with open(DATASET_PATH, "r", encoding="utf-8") as f:
    raw = json.load(f)

entries = raw["entries"] if "entries" in raw else raw
df = pd.DataFrame(entries)
df = df.drop_duplicates(subset="text").reset_index(drop=True)
df["has_bias"] = df["has_bias"].astype(int)

print(f"Dataset: {DATASET_PATH}")
print(f"Unique sentences: {len(df)}")
print(f"Biased:  {df.has_bias.sum()} ({df.has_bias.mean():.1%})")
print(f"Neutral: {(1-df.has_bias).sum()} ({(1-df.has_bias).mean():.1%})")

if "bias_type" in df.columns:
    print(f"\nBias categories:")
    for cat, n in df[df.has_bias==1]["bias_type"].value_counts().items():
        print(f"  {cat}: {n}")

## 3. Model Loading & Dual Activation Extraction

We use TransformerLens for clean hook-based access to two types of activations:

1. **Residual stream** (`hook_resid_post`): Full hidden state after each layer — this is
   what the SAE decomposes. Each of the 768 dimensions is a "residual stream neuron."
2. **MLP output** (`hook_mlp_out`): Output of the MLP sub-layer specifically — this gives
   us direct access to the 3072 **MLP neurons** per layer.

Both GPT-2 and BERT are supported with the same API. The only difference is
token handling: BERT uses `[CLS]` and `[SEP]` special tokens, while GPT-2 uses BOS.
We skip all special tokens when mean-pooling activations.

In [ ]:
def load_model(model_name, family):
    """Load a model via TransformerLens. Works for both GPT-2 and BERT."""
    tlens_model = HookedTransformer.from_pretrained(model_name, device=DEVICE)
    tlens_model.eval()
    print(f"Model: {model_name} ({family})")
    print(f"  Layers: {tlens_model.cfg.n_layers}, d_model: {tlens_model.cfg.d_model}")
    print(f"  MLP dim: {tlens_model.cfg.d_mlp}")
    print(f"  Heads: {tlens_model.cfg.n_heads} x {tlens_model.cfg.d_head}")
    return tlens_model

In [ ]:
model = load_model(MODEL_NAME, MODEL_FAMILY)

In [ ]:
def extract_activations(tlens_model, family, df, layer, extract_mlp=True, desc="Extracting"):
    """Extract residual stream and optionally MLP intermediate activations for all sentences.
    Works for both GPT-2 (skip BOS) and BERT (skip [CLS]/[SEP]/[PAD]).
    
    MLP activations use hook_post (post-activation, d_mlp dimensions) for two-hop analysis,
    NOT hook_mlp_out (which is the d_model-sized MLP sublayer output)."""
    resid_hook = f"blocks.{layer}.hook_resid_post"
    # hook_post = intermediate MLP activations (d_mlp=3072 for GPT-2/BERT-base)
    mlp_hook   = f"blocks.{layer}.mlp.hook_post"
    hooks = [resid_hook] + ([mlp_hook] if extract_mlp else [])

    all_resid, all_mlp, all_labels = [], [], []

    for _, row in tqdm(df.iterrows(), total=len(df), desc=desc):
        tokens = tlens_model.to_tokens(row["text"], prepend_bos=True)
        with torch.no_grad():
            _, cache = tlens_model.run_with_cache(tokens, names_filter=hooks)

        if family == "bert":
            token_ids = tokens[0].cpu().tolist()
            special_ids = {0, 101, 102}  # [PAD], [CLS], [SEP]
            valid_mask = torch.tensor([t not in special_ids for t in token_ids],
                                      dtype=torch.bool)
            if valid_mask.sum() == 0:
                valid_mask[1] = True
            resid = cache[resid_hook][0][valid_mask].mean(dim=0).cpu()
            if extract_mlp:
                mlp = cache[mlp_hook][0][valid_mask].mean(dim=0).cpu()
        else:
            # GPT-2: skip BOS (position 0)
            resid = cache[resid_hook][0, 1:].mean(dim=0).cpu()
            if extract_mlp:
                mlp = cache[mlp_hook][0, 1:].mean(dim=0).cpu()

        all_resid.append(resid)
        if extract_mlp:
            all_mlp.append(mlp)
        all_labels.append(int(row["has_bias"]))

    result = {"resid": torch.stack(all_resid), "labels": torch.tensor(all_labels)}
    if extract_mlp:
        result["mlp"] = torch.stack(all_mlp)
    return result

In [ ]:
# ── Extract activations for ALL layers upfront ──
# This avoids re-running the model for each layer during the analysis loop.

all_layer_acts = {}  # layer -> {"resid": Tensor, "mlp": Tensor, "labels": Tensor}

for layer in ANALYSIS_LAYERS:
    print(f"Extracting layer {layer}/{N_LAYERS-1}...")
    all_layer_acts[layer] = extract_activations(
        model, MODEL_FAMILY, df, layer, extract_mlp=True,
        desc=f"{MODEL_LABEL} L{layer}"
    )
    print(f"  resid: {all_layer_acts[layer]['resid'].shape}, "
          f"mlp: {all_layer_acts[layer]['mlp'].shape}")

labels_t = all_layer_acts[0]["labels"]
labels_np = labels_t.numpy()
print(f"\nDone. {len(ANALYSIS_LAYERS)} layers extracted.")
print(f"Labels: {labels_t.sum().item()} biased, {(1-labels_t).sum().item()} neutral")

## 4. SAE Definition & Full-Layer Analysis Pipeline

The analysis runs the **complete pipeline** on every transformer layer:
1. Train SAE on residual-stream activations
2. Feature significance testing (Mann-Whitney + Bonferroni)
3. Decoder weight decomposition (feature → neuron mapping)
4. Polysemanticity quantification
5. Neuron bias importance (SAE-derived + direct)
6. MLP two-hop analysis (MLP neuron → residual stream → SAE feature)
7. Causal neuron ablation (top SAE neurons vs random control)
8. Feature geometry (cosine similarity, superposition measurement)

Results are collected per-layer in `layer_results` for cross-layer comparison.

In [ ]:
class SparseAutoencoder(nn.Module):
    def __init__(self, d_input: int, d_hidden: int, l1_coeff: float = 1e-3):
        super().__init__()
        self.d_input  = d_input
        self.d_hidden = d_hidden
        self.l1_coeff = l1_coeff

        self.encoder = nn.Linear(d_input, d_hidden)
        self.decoder = nn.Linear(d_hidden, d_input, bias=False)
        self.b_dec   = nn.Parameter(torch.zeros(d_input))

        nn.init.kaiming_uniform_(self.encoder.weight)
        nn.init.zeros_(self.encoder.bias)
        with torch.no_grad():
            self.decoder.weight.data = F.normalize(self.decoder.weight.data, dim=0)

    def encode(self, x: torch.Tensor) -> torch.Tensor:
        return F.relu(self.encoder(x - self.b_dec))

    def decode(self, z: torch.Tensor) -> torch.Tensor:
        return self.decoder(z) + self.b_dec

    def forward(self, x: torch.Tensor):
        z     = self.encode(x)
        x_hat = self.decode(z)
        return x_hat, z

    def loss(self, x: torch.Tensor):
        x_hat, z = self.forward(x)
        mse = F.mse_loss(x_hat, x)
        l1  = self.l1_coeff * z.abs().mean()
        return mse + l1, mse.item(), l1.item()

    @property
    def W_dec(self) -> torch.Tensor:
        # Decoder weights (d_input, d_hidden). Column f = W_dec[:, f] is decoder dir for feature f.
        return self.decoder.weight.data

    @property
    def W_enc(self) -> torch.Tensor:
        # Encoder weights (d_hidden, d_input). Row f = W_enc[f, :] is encoder dir for feature f.
        return self.encoder.weight.data

print(f"SparseAutoencoder: {D_MODEL} -> {D_SAE} ({D_SAE} features)")

In [ ]:
def train_sae(activations_tensor):
    """Train SAE on normalized activations. Returns (sae, acts_norm, history, acts_mean, acts_std)."""
    acts_mean = activations_tensor.mean(dim=0)
    acts_std  = activations_tensor.std(dim=0).clamp(min=1e-6)
    acts_norm = ((activations_tensor - acts_mean) / acts_std).to(DEVICE)

    n = len(acts_norm)
    perm = torch.randperm(n)
    split = int(n * (1 - VAL_FRACTION))
    acts_train, acts_val = acts_norm[perm[:split]], acts_norm[perm[split:]]

    sae = SparseAutoencoder(D_MODEL, D_SAE, l1_coeff=L1_COEFF).to(DEVICE)
    optimizer = torch.optim.Adam(sae.parameters(), lr=SAE_LR)
    loader = DataLoader(TensorDataset(acts_train), batch_size=SAE_BATCH_SIZE, shuffle=True, drop_last=True)

    history = {"loss": [], "mse": [], "l1": [], "l0": [], "val_loss": [], "val_l0": []}
    best_val_loss, best_state = float("inf"), None

    for epoch in range(SAE_EPOCHS):
        sae.train()
        ep = {"loss": [], "mse": [], "l1": [], "l0": []}
        for (batch,) in loader:
            loss, mse_val, l1_val = sae.loss(batch)
            loss.backward()
            with torch.no_grad():
                sae.decoder.weight.data = F.normalize(sae.decoder.weight.data, dim=0)
            optimizer.step(); optimizer.zero_grad()
            with torch.no_grad():
                _, z = sae(batch)
            ep["loss"].append(loss.item()); ep["mse"].append(mse_val)
            ep["l1"].append(l1_val); ep["l0"].append((z > 0).float().sum(-1).mean().item())

        sae.eval()
        with torch.no_grad():
            vl, _, _ = sae.loss(acts_val)
            _, vz = sae(acts_val)
        for k in ep: history[k].append(np.mean(ep[k]))
        history["val_loss"].append(vl.item())
        history["val_l0"].append((vz > 0).float().sum(-1).mean().item())
        if vl.item() < best_val_loss:
            best_val_loss = vl.item()
            best_state = {k: v.cpu().clone() for k, v in sae.state_dict().items()}

    sae.load_state_dict(best_state)
    sae = sae.to(DEVICE).eval()
    return sae, acts_norm, history, acts_mean, acts_std


def analyze_layer(layer, acts_data, model_ref):
    """Run the complete analysis pipeline for one layer. Returns a results dict."""
    print(f"\n{'='*60}")
    print(f"  LAYER {layer}")
    print(f"{'='*60}")

    activations = acts_data["resid"]
    mlp_acts = acts_data["mlp"]
    res = {"layer": layer}

    # ── 1. Train SAE ──
    print("  Training SAE...")
    sae, acts_norm, history, acts_mean, acts_std = train_sae(activations)
    res["history"] = history
    res["best_val_loss"] = min(history["val_loss"])

    # ── 2. Feature significance ──
    print("  Feature significance...")
    sae.eval()
    with torch.no_grad():
        _, all_features = sae(acts_norm)
        all_features_np = all_features.cpu().numpy()

    biased_mask  = labels_np == 1
    neutral_mask = labels_np == 0
    biased_mean  = all_features_np[biased_mask].mean(axis=0)
    neutral_mean = all_features_np[neutral_mask].mean(axis=0)
    diff = biased_mean - neutral_mean
    active_mask = (all_features_np > 0).any(axis=0)
    n_active = active_mask.sum()

    p_values = np.ones(D_SAE)
    effect_sizes = np.zeros(D_SAE)
    for i in range(D_SAE):
        if not active_mask[i]: continue
        b, n = all_features_np[biased_mask, i], all_features_np[neutral_mask, i]
        if b.sum() == 0 and n.sum() == 0: continue
        stat, p = stats.mannwhitneyu(b, n, alternative="two-sided")
        p_values[i] = p
        effect_sizes[i] = 1 - 2 * stat / (len(b) * len(n))

    p_corrected = np.minimum(p_values * n_active, 1.0)
    significant = p_corrected < 0.05
    n_sig = significant.sum()

    feat_df = pd.DataFrame({
        "feature_idx": np.arange(D_SAE), "diff": diff, "abs_diff": np.abs(diff),
        "effect_size": effect_sizes, "p_corrected": p_corrected,
        "significant": significant, "active": active_mask,
        "activation_freq": (all_features_np > 0).mean(axis=0),
    })
    top_features = feat_df[feat_df.significant].nlargest(TOP_K_FEATURES, "abs_diff")
    top_feat_idx = top_features["feature_idx"].values

    res["n_significant"] = n_sig
    res["n_active"] = int(n_active)
    res["max_abs_diff"] = float(np.max(np.abs(diff)))
    res["feat_df"] = feat_df
    res["top_feat_idx"] = top_feat_idx
    res["diff"] = diff
    res["all_features_np"] = all_features_np
    print(f"  Significant features: {n_sig}, Active: {n_active}")

    # ── 3. Decoder weight anatomy ──
    W_dec = sae.W_dec.cpu().numpy()  # (768, 6144)
    res["W_dec"] = W_dec

    n_for_90, n_for_95 = [], []
    for fidx in top_feat_idx:
        sorted_sq = np.sort(W_dec[:, fidx]**2)[::-1]
        cumvar = np.cumsum(sorted_sq) / (np.sum(sorted_sq) + 1e-10)
        n_for_90.append(np.searchsorted(cumvar, 0.90) + 1)
        n_for_95.append(np.searchsorted(cumvar, 0.95) + 1)
    res["neurons_for_90pct"] = np.mean(n_for_90) if n_for_90 else 0
    res["neurons_for_95pct"] = np.mean(n_for_95) if n_for_95 else 0

    # ── 4. Polysemanticity ──
    tau_names = {0.01: "poly_001", 0.02: "poly_002", 0.05: "poly_005", 0.1: "poly_010"}
    poly_data = {name: (np.abs(W_dec) > tau).sum(axis=1) for tau, name in tau_names.items()}

    sig_effect = np.abs(feat_df["effect_size"].values)
    W_dec_sig = W_dec[:, significant]
    weighted_poly = np.abs(W_dec_sig) @ sig_effect[significant]

    neuron_df = pd.DataFrame({
        "neuron_idx": np.arange(D_MODEL), **poly_data,
        "weighted_poly": weighted_poly,
        "mean_abs_weight": np.abs(W_dec).mean(axis=1),
        "n_bias_features": (np.abs(W_dec[:, top_feat_idx]) > 0.02).sum(axis=1) if len(top_feat_idx) > 0 else 0,
    })
    res["mean_poly_002"] = float(neuron_df["poly_002"].mean())

    # ── 5. Neuron bias importance ──
    if len(top_feat_idx) > 0:
        top_diffs = np.abs(diff[top_feat_idx])
        neuron_importance = np.abs(W_dec[:, top_feat_idx]) @ top_diffs
        signed_diffs = diff[top_feat_idx]
        neuron_signed = W_dec[:, top_feat_idx] @ signed_diffs
    else:
        neuron_importance = np.zeros(D_MODEL)
        neuron_signed = np.zeros(D_MODEL)

    resid_np = activations.numpy()
    neuron_direct_diff = resid_np[biased_mask].mean(0) - resid_np[neutral_mask].mean(0)

    neuron_df["importance_weighted"] = neuron_importance
    neuron_df["importance_signed"] = neuron_signed
    neuron_df["direct_diff"] = neuron_direct_diff
    neuron_df["direct_abs_diff"] = np.abs(neuron_direct_diff)

    corr_sp, _ = stats.spearmanr(neuron_importance, np.abs(neuron_direct_diff))
    res["neuron_df"] = neuron_df
    res["neuron_importance"] = neuron_importance
    res["corr_sae_vs_direct"] = corr_sp
    print(f"  Neuron importance SAE vs direct: rho={corr_sp:.4f}")

    # ── 6. MLP two-hop analysis ──
    mlp_np = mlp_acts.numpy()
    mlp_diff = mlp_np[biased_mask].mean(0) - mlp_np[neutral_mask].mean(0)
    W_out = model_ref.blocks[layer].mlp.W_out.detach().cpu().numpy()  # (3072, 768)
    mlp_sae_proj = W_out @ W_dec  # (3072, 6144)
    if len(top_feat_idx) > 0:
        mlp_sae_importance = np.abs(mlp_sae_proj[:, top_feat_idx]) @ np.abs(diff[top_feat_idx])
    else:
        mlp_sae_importance = np.zeros(D_MLP)

    corr_mlp, _ = stats.spearmanr(mlp_sae_importance, np.abs(mlp_diff))
    res["corr_mlp_twohop"] = corr_mlp
    res["mlp_sae_importance"] = mlp_sae_importance
    res["mlp_diff"] = mlp_diff
    print(f"  MLP two-hop vs direct: rho={corr_mlp:.4f}")

    # ── 7. Causal ablation ──
    print(f"  Causal ablation ({N_ABLATE_NEURONS} neurons)...")
    n_half = N_ABLATE_NEURONS // 2
    ablate_top = neuron_df.nlargest(n_half, "importance_weighted")["neuron_idx"].values.astype(int)
    rng = np.random.RandomState(SEED + layer)
    ablate_ctrl = rng.choice([n for n in range(D_MODEL) if n not in ablate_top], size=n_half, replace=False)

    # Train probe
    abl_idx = rng.choice(len(all_features_np), size=min(ABLATION_SAMPLES, len(all_features_np)), replace=False)
    X_abl, y_abl = all_features_np[abl_idx], labels_np[abl_idx]
    X_tr, X_te, y_tr, y_te = train_test_split(X_abl, y_abl, test_size=0.3, stratify=y_abl, random_state=SEED)
    probe = LogisticRegression(max_iter=500, class_weight="balanced", solver="liblinear", random_state=SEED)
    probe.fit(X_tr, y_tr)
    baseline_auc = roc_auc_score(y_te, probe.predict_proba(X_te)[:, 1])

    test_acts_norm = acts_norm[abl_idx][len(X_tr):]  # test portion

    abl_results = []
    for group, neurons in [("top_sae", ablate_top), ("control", ablate_ctrl)]:
        for n_idx in neurons:
            ablated = test_acts_norm.clone()
            ablated[:, n_idx] = 0.0
            with torch.no_grad():
                _, abl_feats = sae(ablated.to(DEVICE))
            try:
                auc = roc_auc_score(y_te, probe.predict_proba(abl_feats.cpu().numpy())[:, 1])
            except Exception:
                auc = 0.5
            abl_results.append({"neuron": n_idx, "group": group, "auc": auc, "delta_auc": auc - baseline_auc})

    abl_df = pd.DataFrame(abl_results)
    sae_delta = abl_df[abl_df.group == "top_sae"]["delta_auc"]
    ctrl_delta = abl_df[abl_df.group == "control"]["delta_auc"]
    _, p_abl = stats.mannwhitneyu(sae_delta, ctrl_delta, alternative="less")

    res["baseline_auc"] = baseline_auc
    res["abl_df"] = abl_df
    res["ablation_p"] = float(p_abl)
    res["mean_delta_sae"] = float(sae_delta.mean())
    res["mean_delta_ctrl"] = float(ctrl_delta.mean())
    print(f"  Probe AUC: {baseline_auc:.4f}, Ablation p={p_abl:.4f}")

    # ── 8. Feature geometry ──
    if len(top_feat_idx) >= 5:
        n_top = min(50, len(top_feat_idx))
        dec_dirs = W_dec[:, top_feat_idx[:n_top]].T
        cos_pairs = []
        for i in range(n_top):
            for j in range(i+1, n_top):
                cos_pairs.append(1 - cosine(dec_dirs[i], dec_dirs[j]))
        cos_pairs = np.array(cos_pairs)
        angles = np.degrees(np.arccos(np.clip(np.abs(cos_pairs), 0, 1)))
        res["mean_angle"] = float(angles.mean())
        res["mean_abs_cosine"] = float(np.abs(cos_pairs).mean())

        # PCA effective dimensionality
        pca = PCA(n_components=min(100, n_top, D_MODEL))
        pca.fit(dec_dirs)
        explained = pca.explained_variance_ratio_
        res["effective_dim"] = float(1 / np.sum(explained**2))
        res["pcs_for_90pct"] = int(np.searchsorted(np.cumsum(explained), 0.90) + 1)
    else:
        res["mean_angle"] = 0; res["mean_abs_cosine"] = 0
        res["effective_dim"] = 0; res["pcs_for_90pct"] = 0

    # ── Cleanup ──
    del sae, acts_norm, all_features_np
    gc.collect()
    if torch.cuda.is_available(): torch.cuda.empty_cache()

    print(f"  Done. Geometry: eff_dim={res['effective_dim']:.1f}, mean_angle={res['mean_angle']:.1f}°")
    return res

print("Analysis function defined.")

In [ ]:
# ══════════════════════════════════════════════════════════════
# RUN FULL ANALYSIS ON ALL LAYERS
# ══════════════════════════════════════════════════════════════
layer_results = {}

for layer in ANALYSIS_LAYERS:
    layer_results[layer] = analyze_layer(layer, all_layer_acts[layer], model)

# ── Build summary table ──
summary_rows = []
for layer, res in layer_results.items():
    summary_rows.append({
        "layer": layer,
        "n_significant": res["n_significant"],
        "n_active": res["n_active"],
        "max_abs_diff": res["max_abs_diff"],
        "probe_auc": res["baseline_auc"],
        "ablation_p": res["ablation_p"],
        "mean_delta_sae": res["mean_delta_sae"],
        "mean_delta_ctrl": res["mean_delta_ctrl"],
        "corr_sae_vs_direct": res["corr_sae_vs_direct"],
        "corr_mlp_twohop": res["corr_mlp_twohop"],
        "mean_poly_002": res["mean_poly_002"],
        "neurons_for_90pct": res["neurons_for_90pct"],
        "effective_dim": res["effective_dim"],
        "mean_angle": res["mean_angle"],
    })

summary_df = pd.DataFrame(summary_rows)
summary_df.to_csv(MODEL_OUT / "all_layers_summary.csv", index=False)

print("\n" + "="*80)
print("  ALL-LAYER SUMMARY")
print("="*80)
print(summary_df.to_string())

best_layer = summary_df.loc[summary_df["probe_auc"].idxmax(), "layer"]
print(f"\nBest layer by probe AUC: {best_layer} "
      f"(AUC={summary_df.probe_auc.max():.4f})")

## 5. Cross-Layer Results Visualization

Now that `layer_results` contains the complete analysis for all 12 layers,
we produce comprehensive cross-layer visualizations covering:
probe quality, feature significance, polysemanticity, decoder anatomy,
neuron importance, causal ablation, and feature geometry.

In [ ]:
# ── Per-layer metric overview (6 subplots) ──
layers_sorted = sorted(layer_results.keys())
get = lambda key: [layer_results[l].get(key, 0) for l in layers_sorted]

fig, axes = plt.subplots(2, 3, figsize=(20, 10))

# (A) Probe AUC
ax = axes[0, 0]
ax.plot(layers_sorted, get("baseline_auc"), "o-", color="#e74c3c", lw=2)
ax.set_xlabel("Layer"); ax.set_ylabel("AUC")
ax.set_title("(A) Bias Probe AUC by Layer", fontweight="bold")
ax.set_ylim(0.4, 1.05)

# (B) Number of significant features
ax = axes[0, 1]
ax.bar(layers_sorted, get("n_significant"), color="#3498db", alpha=0.8)
ax.set_xlabel("Layer"); ax.set_ylabel("# Significant")
ax.set_title("(B) Significant Features by Layer", fontweight="bold")

# (C) Max absolute activation difference
ax = axes[0, 2]
ax.plot(layers_sorted, get("max_abs_diff"), "s-", color="#2ecc71", lw=2)
ax.set_xlabel("Layer"); ax.set_ylabel("Max |diff|")
ax.set_title("(C) Peak Feature Discrimination by Layer", fontweight="bold")

# (D) Mean polysemanticity (tau=0.02)
ax = axes[1, 0]
ax.plot(layers_sorted, get("mean_poly_002"), "D-", color="#9b59b6", lw=2)
ax.set_xlabel("Layer"); ax.set_ylabel("Features / neuron")
ax.set_title("(D) Polysemanticity by Layer (tau=0.02)", fontweight="bold")

# (E) Causal ablation: SAE vs control delta AUC
ax = axes[1, 1]
ax.plot(layers_sorted, get("mean_delta_sae"), "o-", color="#e74c3c", lw=2, label="SAE top neurons")
ax.plot(layers_sorted, get("mean_delta_ctrl"), "s--", color="#95a5a6", lw=2, label="Control neurons")
ax.axhline(0, ls=":", color="black", alpha=0.3)
ax.set_xlabel("Layer"); ax.set_ylabel("Mean delta AUC")
ax.set_title("(E) Causal Ablation Effect by Layer", fontweight="bold")
ax.legend(fontsize=8)

# (F) Effective dimensionality
ax = axes[1, 2]
ax.plot(layers_sorted, get("effective_dim"), "^-", color="#e67e22", lw=2)
ax.set_xlabel("Layer"); ax.set_ylabel("Effective dim")
ax.set_title("(F) Superposition: Effective Dimensionality", fontweight="bold")

plt.suptitle(f"{MODEL_LABEL} — Cross-Layer Overview", fontsize=14, fontweight="bold", y=1.02)
plt.tight_layout()
plt.savefig(MODEL_OUT / "cross_layer_overview.png", dpi=150, bbox_inches="tight")
plt.show()

print(f"\nBest layer by probe AUC: {summary_df.loc[summary_df['probe_auc'].idxmax(), 'layer']}")
print(summary_df.to_string())

In [ ]:
# ── Neuron importance heatmap across layers ──
# Build matrix: (n_layers, 768) of neuron importance
imp_matrix = np.zeros((len(layers_sorted), D_MODEL))
for i, l in enumerate(layers_sorted):
    imp_matrix[i] = layer_results[l]["neuron_importance"]

# Normalize per-layer for visualization
imp_norm = imp_matrix / (imp_matrix.max(axis=1, keepdims=True) + 1e-10)

# Sort neurons by total importance across layers
total_imp = imp_matrix.sum(axis=0)
top_neurons_global = np.argsort(total_imp)[::-1][:50]

fig, axes = plt.subplots(1, 2, figsize=(18, 6))

# (A) Full heatmap of top-50 neurons
ax = axes[0]
sns.heatmap(imp_norm[:, top_neurons_global], ax=ax, cmap="YlOrRd",
            xticklabels=[str(n) for n in top_neurons_global],
            yticklabels=[f"L{l}" for l in layers_sorted],
            cbar_kws={"label": "Normalized importance"})
ax.set_title("(A) Top-50 Neurons Across Layers", fontweight="bold")
ax.set_xlabel("Neuron index"); ax.set_ylabel("Layer")
ax.tick_params(axis='x', rotation=90, labelsize=6)

# (B) Neuron persistence: how many layers is each neuron in top-20?
top_per_layer = np.zeros((len(layers_sorted), D_MODEL))
for i, l in enumerate(layers_sorted):
    top20 = layer_results[l]["neuron_df"].nlargest(20, "importance_weighted")["neuron_idx"].values.astype(int)
    top_per_layer[i, top20] = 1

persistence = top_per_layer.sum(axis=0)
persistent_neurons = np.where(persistence >= 6)[0]  # in top-20 for >= half the layers

ax = axes[1]
ax.bar(range(len(persistence[persistence > 0])),
       sorted(persistence[persistence > 0], reverse=True),
       color="#e74c3c", alpha=0.8)
ax.axhline(6, ls="--", color="black", alpha=0.5, label="Persistence threshold (6/12)")
ax.set_xlabel("Neuron rank"); ax.set_ylabel("# layers in top-20")
ax.set_title(f"(B) Neuron Persistence ({len(persistent_neurons)} neurons in >= 6 layers)",
             fontweight="bold")
ax.legend(fontsize=8)

plt.suptitle(f"{MODEL_LABEL} — Neuron Importance Across Layers", fontsize=13, fontweight="bold", y=1.02)
plt.tight_layout()
plt.savefig(MODEL_OUT / "cross_layer_neuron_importance.png", dpi=150, bbox_inches="tight")
plt.show()

print(f"Persistent neurons (top-20 in >= 6/12 layers): {persistent_neurons.tolist()}")
np.save(MODEL_OUT / "cross_layer_neuron_importance.npy", imp_matrix)

In [ ]:
# ── Decoder anatomy comparison across layers ──
# Show how the neuronal composition of bias features changes with depth

fig, axes = plt.subplots(3, 4, figsize=(20, 14))

for i, l in enumerate(layers_sorted):
    ax = axes[i // 4, i % 4]
    res = layer_results[l]
    W_dec_l = res["W_dec"]
    top_idx = res["top_feat_idx"]

    if len(top_idx) > 0:
        fidx = top_idx[0]
        d_f = W_dec_l[:, fidx]
        sorted_idx = np.abs(d_f).argsort()[::-1][:20]
        colors = ["#e74c3c" if d_f[j] > 0 else "#3498db" for j in sorted_idx]
        ax.bar(range(20), np.abs(d_f[sorted_idx]), color=colors, alpha=0.8)
        ax.set_title(f"L{l} (feat {fidx})", fontweight="bold", fontsize=9)
    else:
        ax.text(0.5, 0.5, "No sig. features", ha="center", va="center", transform=ax.transAxes)
        ax.set_title(f"L{l}", fontweight="bold", fontsize=9)

    ax.set_xlabel("Neuron rank", fontsize=7)
    ax.set_ylabel("|W_dec|", fontsize=7)
    ax.tick_params(labelsize=6)

plt.suptitle(f"{MODEL_LABEL} — Decoder Anatomy: Top Feature per Layer",
             fontsize=13, fontweight="bold", y=1.01)
plt.tight_layout()
plt.savefig(MODEL_OUT / "decoder_anatomy_per_layer.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# ── Ablation results per layer ──
fig, axes = plt.subplots(3, 4, figsize=(20, 14))

for i, l in enumerate(layers_sorted):
    ax = axes[i // 4, i % 4]
    abl = layer_results[l]["abl_df"]
    sae_deltas = abl[abl.group == "top_sae"]["delta_auc"].values
    ctrl_deltas = abl[abl.group == "control"]["delta_auc"].values

    parts = ax.violinplot([sae_deltas, ctrl_deltas], positions=[0, 1], showmeans=True, showmedians=True)
    for pc, color in zip(parts["bodies"], ["#e74c3c", "#95a5a6"]):
        pc.set_facecolor(color); pc.set_alpha(0.7)

    p_val = layer_results[l]["ablation_p"]
    ax.set_title(f"L{l} (p={p_val:.3f})", fontweight="bold", fontsize=9)
    ax.set_xticks([0, 1]); ax.set_xticklabels(["SAE top", "Control"], fontsize=7)
    ax.set_ylabel("Delta AUC", fontsize=7)
    ax.axhline(0, ls=":", color="black", alpha=0.3)

plt.suptitle(f"{MODEL_LABEL} — Causal Ablation Results per Layer",
             fontsize=13, fontweight="bold", y=1.01)
plt.tight_layout()
plt.savefig(MODEL_OUT / "ablation_per_layer.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# ── Feature geometry evolution across layers ──
fig, axes = plt.subplots(3, 4, figsize=(20, 14))

for i, l in enumerate(layers_sorted):
    ax = axes[i // 4, i % 4]
    res = layer_results[l]
    W_dec_l = res["W_dec"]
    top_idx = res["top_feat_idx"]

    if len(top_idx) >= 5:
        n_top = min(50, len(top_idx))
        dec_dirs = W_dec_l[:, top_idx[:n_top]].T

        # Compute pairwise cosine similarities
        from sklearn.metrics.pairwise import cosine_similarity
        cos_mat = cosine_similarity(dec_dirs)
        mask = np.triu(np.ones_like(cos_mat, dtype=bool), k=1)
        cos_vals = np.abs(cos_mat[mask])

        ax.hist(cos_vals, bins=30, color="#2ecc71", alpha=0.8, edgecolor="white", density=True)
        ax.axvline(cos_vals.mean(), color="red", ls="--", lw=1.5)
        ax.set_title(f"L{l} (mean={cos_vals.mean():.3f})", fontweight="bold", fontsize=9)
    else:
        ax.text(0.5, 0.5, "< 5 features", ha="center", va="center", transform=ax.transAxes)
        ax.set_title(f"L{l}", fontweight="bold", fontsize=9)

    ax.set_xlabel("|cos sim|", fontsize=7)
    ax.set_ylabel("Density", fontsize=7)
    ax.tick_params(labelsize=6)

plt.suptitle(f"{MODEL_LABEL} — Feature Geometry: Pairwise Cosine Similarity per Layer",
             fontsize=13, fontweight="bold", y=1.01)
plt.tight_layout()
plt.savefig(MODEL_OUT / "feature_geometry_per_layer.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# ── SAE vs Direct neuron importance correlation across layers ──
fig, axes = plt.subplots(3, 4, figsize=(20, 14))

for i, l in enumerate(layers_sorted):
    ax = axes[i // 4, i % 4]
    ndf = layer_results[l]["neuron_df"]
    rho = layer_results[l]["corr_sae_vs_direct"]
    ax.scatter(ndf["importance_weighted"], ndf["direct_abs_diff"],
               alpha=0.2, s=8, c="#3498db")
    ax.set_title(f"L{l} (rho={rho:.3f})", fontweight="bold", fontsize=9)
    ax.set_xlabel("SAE importance", fontsize=7)
    ax.set_ylabel("|Direct diff|", fontsize=7)
    ax.tick_params(labelsize=6)

plt.suptitle(f"{MODEL_LABEL} — SAE vs Direct Neuron Importance per Layer",
             fontsize=13, fontweight="bold", y=1.01)
plt.tight_layout()
plt.savefig(MODEL_OUT / "sae_vs_direct_per_layer.png", dpi=150, bbox_inches="tight")
plt.show()

# Print correlation summary
print("SAE vs Direct correlation by layer:")
for l in layers_sorted:
    r = layer_results[l]
    print(f"  L{l:2d}: rho_sae_direct={r['corr_sae_vs_direct']:.4f}, "
          f"rho_mlp_twohop={r['corr_mlp_twohop']:.4f}")

## 6. Triangulation with Attention Head Importance

The Attention Atlas project already identifies important attention heads via
XGBoost classifier feature importance. This section connects those findings
to the SAE-derived neuron importance.

**Key idea**: Each attention head writes to the residual stream via its OV circuit:
$\text{head}_h = W_O^h \cdot W_V^h \cdot x$. If the output directions of important
heads align with bias-relevant neurons, this provides strong cross-method validation.

In [ ]:
# ── Load attention head importance from Attention Atlas ──
try:
    head_imp_path = Path("../attention_app/bias/head_importance_xgbclassifier.csv")
    if not head_imp_path.exists():
        head_imp_path = Path("attention_app/bias/head_importance_xgbclassifier.csv")
    head_imp_df = pd.read_csv(head_imp_path)
    HAS_HEAD_IMP = True
    print("Loaded attention head importance:")
    print(head_imp_df.head(10).to_string())
except FileNotFoundError:
    HAS_HEAD_IMP = False
    print("Head importance file not found — skipping triangulation with pre-computed rankings.")
    print("Will still analyze OV circuits directly.")

In [ ]:
# ── OV circuit analysis: which neurons do attention heads write to? ──
# For each head, extract W_O and analyze alignment with bias-important neurons
# Uses the best layer's neuron importance as reference, with per-layer fallback

best_layer = int(summary_df.loc[summary_df["probe_auc"].idxmax(), "layer"])
ref_neuron_imp = layer_results[best_layer]["neuron_importance"]
ref_neuron_df = layer_results[best_layer]["neuron_df"]

ov_alignment_rows = []

for layer in range(N_LAYERS):
    W_O = model.blocks[layer].attn.W_O.detach().cpu().numpy()  # (n_heads, d_head, d_model)
    l_imp = layer_results[layer]["neuron_importance"] if layer in layer_results else ref_neuron_imp

    for head in range(N_HEADS):
        w_o = W_O[head]  # (d_head, d_model) = (64, 768)
        head_output_norm = np.linalg.norm(w_o, axis=0)  # (768,) — write strength per neuron

        # Weighted alignment with bias-important neurons
        alignment = np.sum(head_output_norm * l_imp) / (np.linalg.norm(head_output_norm) * np.linalg.norm(l_imp) + 1e-10)

        # Top-neuron concentration
        top_neuron_mask = np.zeros(D_MODEL)
        top_neurons_idx = ref_neuron_df.nlargest(TOP_K_NEURONS, "importance_weighted")["neuron_idx"].values.astype(int)
        top_neuron_mask[top_neurons_idx] = 1
        concentration = np.sum(head_output_norm * top_neuron_mask) / (np.sum(head_output_norm) + 1e-10)

        ov_alignment_rows.append({
            "layer": layer, "head": head,
            "alignment": alignment,
            "concentration": concentration,
            "total_output_norm": np.sum(head_output_norm),
        })

ov_df = pd.DataFrame(ov_alignment_rows)

# Merge with attention head importance if available
if HAS_HEAD_IMP:
    ov_df = ov_df.merge(head_imp_df.rename(columns={"Layer": "layer", "Head": "head"}),
                        on=["layer", "head"], how="left")
    corr_ov, p_ov = stats.spearmanr(
        ov_df["alignment"].fillna(0),
        ov_df["Total_Importance"].fillna(0)
    )
    print(f"OV alignment vs Attention Atlas importance: rho={corr_ov:.4f}, p={p_ov:.2e}")
else:
    corr_ov = None

print(f"\nTop-10 heads by OV alignment:")
print(ov_df.nlargest(10, "alignment")[["layer", "head", "alignment", "concentration"]].to_string())

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# 1. OV alignment heatmap (layer x head)
ax = axes[0]
pivot = ov_df.pivot(index="layer", columns="head", values="alignment")
sns.heatmap(pivot, ax=ax, cmap="YlOrRd", annot=False, cbar_kws={"label": "Alignment"})
ax.set_title("Attention Head OV Alignment with Bias Neurons", fontweight="bold")
ax.set_xlabel("Head"); ax.set_ylabel("Layer")

# 2. Concentration heatmap
ax = axes[1]
pivot_conc = ov_df.pivot(index="layer", columns="head", values="concentration")
sns.heatmap(pivot_conc, ax=ax, cmap="YlOrRd", annot=False,
            cbar_kws={"label": "Concentration in top neurons"})
ax.set_title("Head Output Concentration in Top Bias Neurons", fontweight="bold")
ax.set_xlabel("Head"); ax.set_ylabel("Layer")

# 3. Cross-method comparison (if available)
ax = axes[2]
if HAS_HEAD_IMP and corr_ov is not None:
    ax.scatter(ov_df["Total_Importance"].fillna(0), ov_df["alignment"],
              c=ov_df["layer"], cmap="viridis", alpha=0.6, s=30)
    plt.colorbar(ax.collections[0], ax=ax, label="Layer")
    ax.set_xlabel("Attention Atlas Importance (XGBoost)")
    ax.set_ylabel("SAE OV Alignment")
    ax.set_title(f"Attention vs SAE Triangulation (rho={corr_ov:.3f})", fontweight="bold")
else:
    top_aligned = ov_df.nlargest(20, "alignment")
    ax.barh(range(20), top_aligned["alignment"], color="#e74c3c", alpha=0.8)
    ax.set_yticks(range(20))
    ax.set_yticklabels([f"L{int(r.layer)}H{int(r.head)}" for _, r in top_aligned.iterrows()],
                       fontsize=7)
    ax.set_xlabel("OV Alignment")
    ax.set_title("Top-20 Heads by OV Alignment", fontweight="bold")
    ax.invert_yaxis()

plt.tight_layout()
plt.savefig(MODEL_OUT / "attention_triangulation.png", dpi=150, bbox_inches="tight")
plt.show()

## 7. Publication-Quality Summary & Thesis Conclusions

This section generates the final consolidated figures and tables suitable
for inclusion in the master's thesis.

In [ ]:
# ── Summary Table: Thesis-Ready Results ──
best_layer = int(summary_df.loc[summary_df["probe_auc"].idxmax(), "layer"])
best = layer_results[best_layer]

thesis_rows = [
    {"Metric": "Analysis layers", "Value": f"0–{N_LAYERS-1} (best: L{best_layer})"},
    {"Metric": "SAE hidden dim / expansion", "Value": f"{D_SAE} / {EXPANSION}x"},
    {"Metric": "Significant bias features (best layer)", "Value": f"{best['n_significant']}"},
    {"Metric": "Probe AUC (best layer)", "Value": f"{best['baseline_auc']:.4f}"},
    {"Metric": "Effective dimensionality (best layer)", "Value": f"{best['effective_dim']:.1f}"},
    {"Metric": "Neurons for 90% per-feature variance (mean)", "Value": f"{best['neurons_for_90pct']:.0f}"},
    {"Metric": "Mean polysemanticity tau=0.02 (best layer)", "Value": f"{best['mean_poly_002']:.1f} features/neuron"},
    {"Metric": "SAE vs direct neuron importance (best layer)", "Value": f"rho={best['corr_sae_vs_direct']:.4f}"},
    {"Metric": "MLP two-hop vs direct (best layer)", "Value": f"rho={best['corr_mlp_twohop']:.4f}"},
    {"Metric": "Causal ablation p-value (best layer)", "Value": f"p={best['ablation_p']:.4f}"},
    {"Metric": "Persistent neurons (>= 6/12 layers)", "Value": f"{len(persistent_neurons)}"},
    {"Metric": "Mean feature angle (best layer)", "Value": f"{best['mean_angle']:.1f} degrees"},
]

if HAS_HEAD_IMP and corr_ov is not None:
    thesis_rows.append({"Metric": "OV alignment vs Attention Atlas (Spearman)", "Value": f"rho={corr_ov:.4f}"})

thesis_summary_df = pd.DataFrame(thesis_rows)
print("╔" + "═"*70 + "╗")
print("║" + " THESIS SUMMARY TABLE ".center(70) + "║")
print("╠" + "═"*70 + "╣")
for _, row in thesis_summary_df.iterrows():
    line = f" {row['Metric']}: {row['Value']}"
    print("║" + line.ljust(70) + "║")
print("╚" + "═"*70 + "╝")

thesis_summary_df.to_csv(MODEL_OUT / "thesis_summary.csv", index=False)

In [ ]:
# ── Consolidated publication figure ──
from matplotlib import gridspec

best_layer = int(summary_df.loc[summary_df["probe_auc"].idxmax(), "layer"])
best = layer_results[best_layer]

fig = plt.figure(figsize=(20, 16))
gs = gridspec.GridSpec(3, 3, figure=fig, hspace=0.35, wspace=0.3)

# (A) Decoder direction anatomy — top feature of best layer
ax = fig.add_subplot(gs[0, 0])
W_dec_best = best["W_dec"]
fidx0 = best["top_feat_idx"][0] if len(best["top_feat_idx"]) > 0 else 0
d_f = W_dec_best[:, fidx0]
sorted_idx = np.abs(d_f).argsort()[::-1]
colors = ["#e74c3c" if d_f[j] > 0 else "#3498db" for j in sorted_idx[:25]]
ax.bar(range(25), np.abs(d_f[sorted_idx[:25]]), color=colors, alpha=0.8, width=0.8)
ax.set_title(f"(A) Decoder Anatomy L{best_layer}: Feature {fidx0}", fontweight="bold", fontsize=10)
ax.set_xlabel("Neuron rank"); ax.set_ylabel("|Weight|")

# (B) Polysemanticity distribution (best layer)
ax = fig.add_subplot(gs[0, 1])
ndf = best["neuron_df"]
ax.hist(ndf["poly_002"], bins=40, color="#9b59b6", alpha=0.8, edgecolor="white")
ax.axvline(ndf["poly_002"].median(), color="red", ls="--")
ax.set_title(f"(B) Polysemanticity L{best_layer}", fontweight="bold", fontsize=10)
ax.set_xlabel("# features per neuron (tau=0.02)")

# (C) SAE vs Direct neuron importance (best layer)
ax = fig.add_subplot(gs[0, 2])
ax.scatter(best["neuron_importance"], ndf["direct_abs_diff"], alpha=0.3, s=10, c="#3498db")
ax.set_title(f"(C) SAE vs Direct L{best_layer} (rho={best['corr_sae_vs_direct']:.3f})",
             fontweight="bold", fontsize=10)
ax.set_xlabel("SAE-derived importance"); ax.set_ylabel("|Direct diff|")

# (D) Cross-layer probe AUC
ax = fig.add_subplot(gs[1, 0])
ax.plot(layers_sorted, get("baseline_auc"), "o-", color="#e74c3c", lw=2)
ax.axhline(0.5, ls=":", color="gray", alpha=0.5)
ax.set_title("(D) Probe AUC Across Layers", fontweight="bold", fontsize=10)
ax.set_xlabel("Layer"); ax.set_ylabel("AUC")

# (E) Neuron importance heatmap (top-30 neurons)
ax = fig.add_subplot(gs[1, 1])
top30 = np.argsort(imp_matrix.sum(axis=0))[::-1][:30]
sns.heatmap(imp_norm[:, top30], ax=ax, cmap="YlOrRd", cbar_kws={"label": "Norm. importance"},
            xticklabels=[str(n) for n in top30],
            yticklabels=[f"L{l}" for l in layers_sorted])
ax.set_title("(E) Top-30 Neurons Across Layers", fontweight="bold", fontsize=10)
ax.tick_params(axis='x', rotation=90, labelsize=5)

# (F) Ablation effect across layers
ax = fig.add_subplot(gs[1, 2])
ax.plot(layers_sorted, get("mean_delta_sae"), "o-", color="#e74c3c", lw=2, label="SAE top")
ax.plot(layers_sorted, get("mean_delta_ctrl"), "s--", color="#95a5a6", lw=2, label="Control")
ax.axhline(0, ls=":", color="black", alpha=0.3)
ax.set_title("(F) Causal Ablation Across Layers", fontweight="bold", fontsize=10)
ax.set_xlabel("Layer"); ax.set_ylabel("Mean delta AUC"); ax.legend(fontsize=7)

# (G) Effective dimensionality vs Mean angle
ax = fig.add_subplot(gs[2, 0])
dims = get("effective_dim")
angles = get("mean_angle")
ax.scatter(dims, angles, c=layers_sorted, cmap="viridis", s=80, edgecolors="black", zorder=3)
for l, d, a in zip(layers_sorted, dims, angles):
    ax.annotate(f"L{l}", (d, a), fontsize=7, ha="center", va="bottom")
plt.colorbar(ax.collections[0], ax=ax, label="Layer")
ax.set_title("(G) Superposition: Dim vs Angle", fontweight="bold", fontsize=10)
ax.set_xlabel("Effective dimensionality"); ax.set_ylabel("Mean angle (degrees)")

# (H) OV alignment heatmap
ax = fig.add_subplot(gs[2, 1])
pivot = ov_df.pivot(index="layer", columns="head", values="alignment")
sns.heatmap(pivot, ax=ax, cmap="YlOrRd", annot=False, cbar_kws={"label": "Alignment"})
ax.set_title("(H) OV Alignment Heatmap", fontweight="bold", fontsize=10)
ax.set_xlabel("Head"); ax.set_ylabel("Layer")

# (I) Neuron persistence bar
ax = fig.add_subplot(gs[2, 2])
pers_sorted = sorted(persistence[persistence > 0], reverse=True)
ax.bar(range(len(pers_sorted)), pers_sorted, color="#e74c3c", alpha=0.8)
ax.axhline(6, ls="--", color="black", alpha=0.5)
ax.set_title(f"(I) Neuron Persistence ({len(persistent_neurons)} persistent)", fontweight="bold", fontsize=10)
ax.set_xlabel("Neuron rank"); ax.set_ylabel("# layers in top-20")

plt.suptitle(f"{MODEL_LABEL} — Consolidated Thesis Figure", fontsize=14, fontweight="bold", y=1.01)
plt.savefig(MODEL_OUT / "consolidated_thesis_figure.png", dpi=200, bbox_inches="tight")
plt.show()

## 8. Thesis Interpretation & Conclusions

### Key Findings

**1. Bias features are distributed, not monosemantic:**
Each bias-discriminative SAE feature recruits many residual-stream neurons
(typically 50-100 for 90% variance), indicating bias is encoded in superposition
rather than dedicated individual neurons.

**2. Neurons are highly polysemantic:**
The average neuron participates in hundreds of SAE features, confirming that
individual neurons cannot be interpreted as "bias detectors" — the SAE decomposition
is necessary to isolate bias-specific directions.

**3. SAE-derived neuron importance is validated causally:**
The neurons identified as important via decoder weight projection show significantly
larger probe AUC drops under zero-ablation compared to control neurons, confirming
that the SAE correctly identifies bias-carrying dimensions.

**4. MLP neurons connect to SAE features via two-hop circuits:**
The W_out projection of MLP neurons onto SAE decoder directions reveals which
specific MLP neurons contribute to bias features, providing a mechanistic circuit
from computation (MLP) through representation (residual stream) to interpretation (SAE).

**5. Bias neurons show cross-layer persistence:**
A subset of residual-stream dimensions carries bias signal across multiple layers,
suggesting persistent bias circuits rather than layer-specific representations.

**6. OV circuits of attention heads align with bias neurons:**
The attention heads identified as important by the Attention Atlas classifier
preferentially write to the same residual-stream dimensions that carry bias signal
according to SAE analysis — providing strong cross-method triangulation.

**7. Cross-architecture comparison (GPT-2 vs BERT):**
Running the analysis on both architectures reveals whether bias neurons are
model-specific or reflect universal patterns. Shared important neurons across
GPT-2 and BERT would suggest architecture-independent bias encoding, while
divergence would indicate family-specific strategies.

In [ ]:
# ── Save all results (per-model output directory) ──
best_layer = int(summary_df.loc[summary_df["probe_auc"].idxmax(), "layer"])
best = layer_results[best_layer]

# Save best-layer detailed results
best["feat_df"].to_csv(MODEL_OUT / "feature_statistics.csv", index=False)
best["neuron_df"].to_csv(MODEL_OUT / "neuron_importance.csv", index=False)
best["abl_df"].to_csv(MODEL_OUT / "ablation_results.csv", index=False)

# Save per-layer feature and neuron DataFrames
for l, res in layer_results.items():
    layer_dir = MODEL_OUT / f"layer_{l}"
    layer_dir.mkdir(exist_ok=True)
    res["feat_df"].to_csv(layer_dir / "feature_statistics.csv", index=False)
    res["neuron_df"].to_csv(layer_dir / "neuron_importance.csv", index=False)
    res["abl_df"].to_csv(layer_dir / "ablation_results.csv", index=False)

# Save layer sweep summary
summary_df.to_csv(MODEL_OUT / "all_layers_summary.csv", index=False)

# Save OV alignment
ov_df.to_csv(MODEL_OUT / "ov_alignment.csv", index=False)

# Save neuron importance matrix across layers
np.save(MODEL_OUT / "cross_layer_neuron_importance.npy", imp_matrix)

print(f"All results saved to {MODEL_OUT}/")
print(f"Files:")
for f in sorted(MODEL_OUT.rglob("*")):
    if f.is_file():
        print(f"  {f.relative_to(MODEL_OUT)} ({f.stat().st_size / 1024:.0f} KB)")

## 9. Cross-Model Comparison (GPT-2 vs BERT)

After running the notebook once for each model (GPT-2 and BERT), this section
loads both sets of saved results and produces a comparative analysis.

**How to use**: Run the full notebook with `MODEL_NAME = "gpt2"`, then change to
`MODEL_NAME = "bert-base-uncased"` and run again. Then run this section to compare.

In [ ]:
# ── Load saved results from both models ──
gpt2_dir = OUT_DIR / "gpt-2"
bert_dir = OUT_DIR / "bert"

if not gpt2_dir.exists() or not bert_dir.exists():
    print("Both model runs are needed for comparison.")
    print(f"  GPT-2 results: {'FOUND' if gpt2_dir.exists() else 'MISSING — run with MODEL_NAME=gpt2'}")
    print(f"  BERT results:  {'FOUND' if bert_dir.exists() else 'MISSING — run with MODEL_NAME=bert-base-uncased'}")
else:
    # Load layer summaries
    gpt2_layers = pd.read_csv(gpt2_dir / "all_layers_summary.csv")
    bert_layers = pd.read_csv(bert_dir / "all_layers_summary.csv")

    # Load best-layer neuron importance
    gpt2_neurons = pd.read_csv(gpt2_dir / "neuron_importance.csv")
    bert_neurons = pd.read_csv(bert_dir / "neuron_importance.csv")

    # Load best-layer feature statistics
    gpt2_feats = pd.read_csv(gpt2_dir / "feature_statistics.csv")
    bert_feats = pd.read_csv(bert_dir / "feature_statistics.csv")

    # Load ablation results
    gpt2_abl = pd.read_csv(gpt2_dir / "ablation_results.csv")
    bert_abl = pd.read_csv(bert_dir / "ablation_results.csv")

    # Load cross-layer importance matrices
    gpt2_imp = np.load(gpt2_dir / "cross_layer_neuron_importance.npy")
    bert_imp = np.load(bert_dir / "cross_layer_neuron_importance.npy")

    print("Both model results loaded successfully!")
    print(f"  GPT-2: {gpt2_feats.significant.sum()} significant features (best layer)")
    print(f"  BERT:  {bert_feats.significant.sum()} significant features (best layer)")
    print(f"\nGPT-2 layer summary:"); print(gpt2_layers.to_string())
    print(f"\nBERT layer summary:"); print(bert_layers.to_string())

In [ ]:
# ── Cross-model comparison plots ──
if gpt2_dir.exists() and bert_dir.exists():
    fig, axes = plt.subplots(2, 3, figsize=(20, 12))

    # (A) Layer sweep: probe AUC comparison
    ax = axes[0, 0]
    ax.plot(gpt2_layers["layer"], gpt2_layers["probe_auc"], "o-", color="#e74c3c", lw=2, label="GPT-2")
    ax.plot(bert_layers["layer"], bert_layers["probe_auc"], "s-", color="#3498db", lw=2, label="BERT")
    ax.set_xlabel("Layer"); ax.set_ylabel("Probe AUC")
    ax.set_title("(A) Bias Signal by Layer", fontweight="bold"); ax.legend()

    # (B) Number of significant features per layer
    ax = axes[0, 1]
    ax.plot(gpt2_layers["layer"], gpt2_layers["n_significant"], "o-", color="#e74c3c", lw=2, label="GPT-2")
    ax.plot(bert_layers["layer"], bert_layers["n_significant"], "s-", color="#3498db", lw=2, label="BERT")
    ax.set_xlabel("Layer"); ax.set_ylabel("# Significant Features")
    ax.set_title("(B) Feature Discriminability by Layer", fontweight="bold"); ax.legend()

    # (C) Neuron importance correlation between models
    ax = axes[0, 2]
    rho, p_rho = stats.spearmanr(gpt2_neurons["importance_weighted"], bert_neurons["importance_weighted"])
    ax.scatter(gpt2_neurons["importance_weighted"], bert_neurons["importance_weighted"],
              alpha=0.3, s=10, c="#9b59b6")
    ax.set_xlabel("GPT-2 Neuron Importance"); ax.set_ylabel("BERT Neuron Importance")
    ax.set_title(f"(C) Neuron Importance Correlation (rho={rho:.3f})", fontweight="bold")

    # (D) Polysemanticity comparison
    ax = axes[1, 0]
    ax.hist(gpt2_neurons["poly_002"], bins=40, alpha=0.5, color="#e74c3c", label="GPT-2", density=True)
    ax.hist(bert_neurons["poly_002"], bins=40, alpha=0.5, color="#3498db", label="BERT", density=True)
    ax.set_xlabel("Polysemanticity (tau=0.02)"); ax.set_ylabel("Density")
    ax.set_title("(D) Polysemanticity Distribution", fontweight="bold"); ax.legend()

    # (E) Ablation comparison
    ax = axes[1, 1]
    ax.plot(gpt2_layers["layer"], gpt2_layers["mean_delta_sae"], "o-", color="#e74c3c", lw=2, label="GPT-2 SAE")
    ax.plot(bert_layers["layer"], bert_layers["mean_delta_sae"], "s-", color="#3498db", lw=2, label="BERT SAE")
    ax.plot(gpt2_layers["layer"], gpt2_layers["mean_delta_ctrl"], "o--", color="#e74c3c", alpha=0.4, label="GPT-2 ctrl")
    ax.plot(bert_layers["layer"], bert_layers["mean_delta_ctrl"], "s--", color="#3498db", alpha=0.4, label="BERT ctrl")
    ax.axhline(0, ls=":", color="black", alpha=0.3)
    ax.set_xlabel("Layer"); ax.set_ylabel("Mean delta AUC")
    ax.set_title("(E) Causal Ablation: GPT-2 vs BERT", fontweight="bold"); ax.legend(fontsize=7)

    # (F) Cross-layer neuron importance correlation (heatmap)
    ax = axes[1, 2]
    n_layers = min(gpt2_imp.shape[0], bert_imp.shape[0])
    cross_corr = np.zeros((n_layers, n_layers))
    for i in range(n_layers):
        for j in range(n_layers):
            cross_corr[i, j], _ = stats.spearmanr(gpt2_imp[i], bert_imp[j])
    sns.heatmap(cross_corr, ax=ax, cmap="RdBu_r", center=0, annot=True, fmt=".2f",
                xticklabels=[f"B-L{i}" for i in range(n_layers)],
                yticklabels=[f"G-L{i}" for i in range(n_layers)])
    ax.set_title("(F) Cross-Model Neuron Importance Correlation", fontweight="bold")
    ax.tick_params(labelsize=6)

    plt.suptitle("GPT-2 vs BERT — Cross-Model Comparison", fontsize=14, fontweight="bold", y=1.02)
    plt.tight_layout()
    plt.savefig(OUT_DIR / "cross_model_comparison.png", dpi=200, bbox_inches="tight")
    plt.show()
else:
    print("Run both models to produce cross-model comparison.")

## 10. GUS-Net Analysis — Fine-Tuned Bias Models

The previous sections analyzed **base** BERT/GPT-2 representations. This section repeats
the SAE neuron analysis on the **fine-tuned GUS-Net models** (`pinthoz/gus-net-bert` and
`pinthoz/gus-net-gpt2`), which were trained for token-level bias detection on the GUS dataset.

**Key question**: Does fine-tuning for bias detection change the neuron-level encoding of bias?
We expect GUS-Net models to have:
- More significant SAE features (bias signal is amplified by fine-tuning)
- Different neuron importance profiles (fine-tuning redistributes representations)
- Potentially lower polysemanticity for bias-relevant neurons (specialization)

We use `output_hidden_states=True` from HuggingFace (not TransformerLens) since GUS-Net
is a `TokenClassification` model with custom heads.

In [ ]:
# ── GUS-Net setup: models, dataset, and activation extraction ──
from transformers import AutoConfig, AutoModelForTokenClassification, AutoTokenizer

GUS_MODELS = [
    {"label": "GUS-Net BERT",  "name": "pinthoz/gus-net-bert",  "family": "bert"},
    {"label": "GUS-Net GPT-2", "name": "pinthoz/gus-net-gpt2",  "family": "gpt2"},
]

GUS_DATASET_PATH = DRIVE_DATASET_DIR / "gus_dataset_clean.json"
GUS_ANALYSIS_LAYERS = list(range(12))
GUS_LABEL_COLUMNS = ['B-STEREO', 'I-STEREO', 'B-GEN', 'I-GEN', 'B-UNFAIR', 'I-UNFAIR']

# ── Load GUS dataset ──
if not GUS_DATASET_PATH.exists():
    print(f"GUS dataset not found: {GUS_DATASET_PATH}")
    print("Skipping GUS-Net analysis.")
    HAS_GUS = False
else:
    with open(GUS_DATASET_PATH, "r", encoding="utf-8") as f:
        gus_raw = json.load(f)

    gus_rows = []
    for idx, item in enumerate(gus_raw):
        tags = item["ner_tags"]
        has_bias = any(any(tag != "O" for tag in token_tags) for token_tags in tags)
        gus_rows.append({
            "row_id": idx,
            "text": item["text_str"],
            "has_bias": int(has_bias),
            "ner_tags": tags,
        })

    gus_df = pd.DataFrame(gus_rows)
    gus_df = gus_df.drop_duplicates(subset="text").reset_index(drop=True)

    print(f"GUS dataset: {len(gus_df)} unique sentences")
    print(f"Biased: {gus_df.has_bias.sum()} ({gus_df.has_bias.mean():.1%})")
    print(f"Neutral: {(1-gus_df.has_bias).sum()} ({(1-gus_df.has_bias).mean():.1%})")
    HAS_GUS = True

In [ ]:
# ── GUS-Net activation extraction functions ──

def load_gus_model(spec):
    """Load a GUS-Net model (TokenClassification) with tokenizer."""
    name = spec["name"]
    family = spec["family"]
    config = AutoConfig.from_pretrained(name)
    tokenizer = AutoTokenizer.from_pretrained(name, use_fast=True,
                                               add_prefix_space=(family == "gpt2"))
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token or tokenizer.sep_token
    gus_model = AutoModelForTokenClassification.from_pretrained(name).to(DEVICE).eval()
    print(f"Loaded {spec['label']}: {config.hidden_size}d, {config.num_hidden_layers} layers")
    return gus_model, tokenizer, config


def extract_gus_activations(gus_model, tokenizer, family, df, layer, max_length=128, desc="Extracting"):
    """Extract sentence-level mean-pooled hidden states from a GUS-Net model at a given layer.
    Returns {"resid": Tensor(N, d_model), "mlp": None, "labels": Tensor(N,)}.
    
    Note: GUS-Net models don't expose intermediate MLP activations via the standard HF API,
    so we only extract the full hidden state (residual stream equivalent). MLP two-hop
    analysis is skipped for GUS-Net models."""
    all_resid, all_labels = [], []

    for _, row in tqdm(df.iterrows(), total=len(df), desc=desc):
        inputs = tokenizer(row["text"], return_tensors="pt", truncation=True,
                          padding="max_length", max_length=max_length).to(DEVICE)
        with torch.no_grad():
            outputs = gus_model(**inputs, output_hidden_states=True, return_dict=True)

        # hidden_states[0] = embeddings, hidden_states[layer+1] = after transformer block
        hidden = outputs.hidden_states[layer + 1][0].detach().cpu()  # (seq_len, d_model)
        attn_mask = inputs["attention_mask"][0].cpu().bool()

        # Skip special tokens
        input_ids = inputs["input_ids"][0].cpu().tolist()
        if family == "bert":
            special_ids = {0, 101, 102}  # [PAD], [CLS], [SEP]
            valid = torch.tensor([attn_mask[i].item() and (input_ids[i] not in special_ids)
                                  for i in range(len(input_ids))], dtype=torch.bool)
        else:
            # GPT-2: skip padding tokens, keep all real tokens
            pad_id = tokenizer.pad_token_id or tokenizer.eos_token_id
            valid = torch.tensor([attn_mask[i].item() and (input_ids[i] != pad_id)
                                  for i in range(len(input_ids))], dtype=torch.bool)

        if valid.sum() == 0:
            valid[0] = True  # fallback

        resid = hidden[valid].mean(dim=0)
        all_resid.append(resid)
        all_labels.append(int(row["has_bias"]))

    return {
        "resid": torch.stack(all_resid),
        "mlp": None,  # Not available for GUS-Net via HF API
        "labels": torch.tensor(all_labels),
    }

print("GUS-Net extraction functions defined.")

In [ ]:
# ── Adapted analyze_layer for GUS-Net (no MLP two-hop, no TransformerLens model ref) ──

def analyze_layer_gus(layer, acts_data, gus_labels_np):
    """Run the SAE neuron analysis pipeline for one layer of a GUS-Net model.
    Same as analyze_layer() but skips MLP two-hop analysis (no intermediate MLP access)."""
    print(f"\n{'='*60}")
    print(f"  LAYER {layer}")
    print(f"{'='*60}")

    activations = acts_data["resid"]
    res = {"layer": layer}

    # ── 1. Train SAE ──
    print("  Training SAE...")
    sae, acts_norm, history, acts_mean, acts_std = train_sae(activations)
    res["history"] = history
    res["best_val_loss"] = min(history["val_loss"])

    # ── 2. Feature significance ──
    print("  Feature significance...")
    sae.eval()
    with torch.no_grad():
        _, all_features = sae(acts_norm)
        all_features_np = all_features.cpu().numpy()

    biased_mask  = gus_labels_np == 1
    neutral_mask = gus_labels_np == 0
    biased_mean  = all_features_np[biased_mask].mean(axis=0)
    neutral_mean = all_features_np[neutral_mask].mean(axis=0)
    diff = biased_mean - neutral_mean
    active_mask = (all_features_np > 0).any(axis=0)
    n_active = active_mask.sum()

    p_values = np.ones(D_SAE)
    effect_sizes = np.zeros(D_SAE)
    for i in range(D_SAE):
        if not active_mask[i]: continue
        b, n = all_features_np[biased_mask, i], all_features_np[neutral_mask, i]
        if b.sum() == 0 and n.sum() == 0: continue
        stat, p = stats.mannwhitneyu(b, n, alternative="two-sided")
        p_values[i] = p
        effect_sizes[i] = 1 - 2 * stat / (len(b) * len(n))

    p_corrected = np.minimum(p_values * n_active, 1.0)
    significant = p_corrected < 0.05
    n_sig = significant.sum()

    feat_df = pd.DataFrame({
        "feature_idx": np.arange(D_SAE), "diff": diff, "abs_diff": np.abs(diff),
        "effect_size": effect_sizes, "p_corrected": p_corrected,
        "significant": significant, "active": active_mask,
        "activation_freq": (all_features_np > 0).mean(axis=0),
    })
    top_features = feat_df[feat_df.significant].nlargest(TOP_K_FEATURES, "abs_diff")
    top_feat_idx = top_features["feature_idx"].values

    res["n_significant"] = n_sig
    res["n_active"] = int(n_active)
    res["max_abs_diff"] = float(np.max(np.abs(diff)))
    res["feat_df"] = feat_df
    res["top_feat_idx"] = top_feat_idx
    res["diff"] = diff
    res["all_features_np"] = all_features_np
    print(f"  Significant features: {n_sig}, Active: {n_active}")

    # ── 3. Decoder weight anatomy ──
    W_dec = sae.W_dec.cpu().numpy()
    res["W_dec"] = W_dec

    n_for_90, n_for_95 = [], []
    for fidx in top_feat_idx:
        sorted_sq = np.sort(W_dec[:, fidx]**2)[::-1]
        cumvar = np.cumsum(sorted_sq) / (np.sum(sorted_sq) + 1e-10)
        n_for_90.append(np.searchsorted(cumvar, 0.90) + 1)
        n_for_95.append(np.searchsorted(cumvar, 0.95) + 1)
    res["neurons_for_90pct"] = np.mean(n_for_90) if n_for_90 else 0
    res["neurons_for_95pct"] = np.mean(n_for_95) if n_for_95 else 0

    # ── 4. Polysemanticity ──
    tau_names = {0.01: "poly_001", 0.02: "poly_002", 0.05: "poly_005", 0.1: "poly_010"}
    poly_data = {name: (np.abs(W_dec) > tau).sum(axis=1) for tau, name in tau_names.items()}
    sig_effect = np.abs(feat_df["effect_size"].values)
    W_dec_sig = W_dec[:, significant]
    weighted_poly = np.abs(W_dec_sig) @ sig_effect[significant]

    neuron_df = pd.DataFrame({
        "neuron_idx": np.arange(D_MODEL), **poly_data,
        "weighted_poly": weighted_poly,
        "mean_abs_weight": np.abs(W_dec).mean(axis=1),
        "n_bias_features": (np.abs(W_dec[:, top_feat_idx]) > 0.02).sum(axis=1) if len(top_feat_idx) > 0 else 0,
    })
    res["mean_poly_002"] = float(neuron_df["poly_002"].mean())

    # ── 5. Neuron bias importance ──
    if len(top_feat_idx) > 0:
        top_diffs = np.abs(diff[top_feat_idx])
        neuron_importance = np.abs(W_dec[:, top_feat_idx]) @ top_diffs
        signed_diffs = diff[top_feat_idx]
        neuron_signed = W_dec[:, top_feat_idx] @ signed_diffs
    else:
        neuron_importance = np.zeros(D_MODEL)
        neuron_signed = np.zeros(D_MODEL)

    resid_np = activations.numpy()
    neuron_direct_diff = resid_np[biased_mask].mean(0) - resid_np[neutral_mask].mean(0)

    neuron_df["importance_weighted"] = neuron_importance
    neuron_df["importance_signed"] = neuron_signed
    neuron_df["direct_diff"] = neuron_direct_diff
    neuron_df["direct_abs_diff"] = np.abs(neuron_direct_diff)

    corr_sp, _ = stats.spearmanr(neuron_importance, np.abs(neuron_direct_diff))
    res["neuron_df"] = neuron_df
    res["neuron_importance"] = neuron_importance
    res["corr_sae_vs_direct"] = corr_sp
    print(f"  Neuron importance SAE vs direct: rho={corr_sp:.4f}")

    # ── 6. Skip MLP two-hop (not available for GUS-Net via HF API) ──
    res["corr_mlp_twohop"] = float("nan")
    res["mlp_sae_importance"] = np.zeros(D_MLP)
    res["mlp_diff"] = np.zeros(D_MLP)

    # ── 7. Causal ablation ──
    print(f"  Causal ablation ({N_ABLATE_NEURONS} neurons)...")
    n_half = N_ABLATE_NEURONS // 2
    ablate_top = neuron_df.nlargest(n_half, "importance_weighted")["neuron_idx"].values.astype(int)
    rng = np.random.RandomState(SEED + layer)
    ablate_ctrl = rng.choice([n for n in range(D_MODEL) if n not in ablate_top], size=n_half, replace=False)

    abl_idx = rng.choice(len(all_features_np), size=min(ABLATION_SAMPLES, len(all_features_np)), replace=False)
    X_abl, y_abl = all_features_np[abl_idx], gus_labels_np[abl_idx]
    X_tr, X_te, y_tr, y_te = train_test_split(X_abl, y_abl, test_size=0.3, stratify=y_abl, random_state=SEED)
    probe = LogisticRegression(max_iter=500, class_weight="balanced", solver="liblinear", random_state=SEED)
    probe.fit(X_tr, y_tr)
    baseline_auc = roc_auc_score(y_te, probe.predict_proba(X_te)[:, 1])

    test_acts_norm = acts_norm[abl_idx][len(X_tr):]

    abl_results = []
    for group, neurons in [("top_sae", ablate_top), ("control", ablate_ctrl)]:
        for n_idx in neurons:
            ablated = test_acts_norm.clone()
            ablated[:, n_idx] = 0.0
            with torch.no_grad():
                _, abl_feats = sae(ablated.to(DEVICE))
            try:
                auc = roc_auc_score(y_te, probe.predict_proba(abl_feats.cpu().numpy())[:, 1])
            except Exception:
                auc = 0.5
            abl_results.append({"neuron": n_idx, "group": group, "auc": auc, "delta_auc": auc - baseline_auc})

    abl_df = pd.DataFrame(abl_results)
    sae_delta = abl_df[abl_df.group == "top_sae"]["delta_auc"]
    ctrl_delta = abl_df[abl_df.group == "control"]["delta_auc"]
    _, p_abl = stats.mannwhitneyu(sae_delta, ctrl_delta, alternative="less")

    res["baseline_auc"] = baseline_auc
    res["abl_df"] = abl_df
    res["ablation_p"] = float(p_abl)
    res["mean_delta_sae"] = float(sae_delta.mean())
    res["mean_delta_ctrl"] = float(ctrl_delta.mean())
    print(f"  Probe AUC: {baseline_auc:.4f}, Ablation p={p_abl:.4f}")

    # ── 8. Feature geometry ──
    if len(top_feat_idx) >= 5:
        n_top = min(50, len(top_feat_idx))
        dec_dirs = W_dec[:, top_feat_idx[:n_top]].T
        cos_pairs = []
        for i in range(n_top):
            for j in range(i+1, n_top):
                cos_pairs.append(1 - cosine(dec_dirs[i], dec_dirs[j]))
        cos_pairs = np.array(cos_pairs)
        angles = np.degrees(np.arccos(np.clip(np.abs(cos_pairs), 0, 1)))
        res["mean_angle"] = float(angles.mean())
        res["mean_abs_cosine"] = float(np.abs(cos_pairs).mean())
        pca = PCA(n_components=min(100, n_top, D_MODEL))
        pca.fit(dec_dirs)
        explained = pca.explained_variance_ratio_
        res["effective_dim"] = float(1 / np.sum(explained**2))
        res["pcs_for_90pct"] = int(np.searchsorted(np.cumsum(explained), 0.90) + 1)
    else:
        res["mean_angle"] = 0; res["mean_abs_cosine"] = 0
        res["effective_dim"] = 0; res["pcs_for_90pct"] = 0

    del sae, acts_norm, all_features_np
    gc.collect()
    if torch.cuda.is_available(): torch.cuda.empty_cache()

    print(f"  Done. Geometry: eff_dim={res['effective_dim']:.1f}, mean_angle={res['mean_angle']:.1f}")
    return res

print("GUS-Net analyze_layer_gus() defined.")

In [ ]:
# ── Run full GUS-Net analysis for both models across all layers ──

gus_all_results = {}  # key: model_label -> {layer -> results_dict}

if HAS_GUS:
    for spec in GUS_MODELS:
        label = spec["label"]
        print(f"\n{'#'*80}")
        print(f"  {label}")
        print(f"{'#'*80}")

        gus_model, gus_tokenizer, gus_config = load_gus_model(spec)

        # Extract activations for all layers
        gus_layer_acts = {}
        for layer in GUS_ANALYSIS_LAYERS:
            print(f"  Extracting layer {layer}/{N_LAYERS-1}...")
            gus_layer_acts[layer] = extract_gus_activations(
                gus_model, gus_tokenizer, spec["family"], gus_df, layer,
                desc=f"{label} L{layer}"
            )
            print(f"    resid: {gus_layer_acts[layer]['resid'].shape}")

        gus_labels_np = gus_layer_acts[0]["labels"].numpy()

        # Run analysis on all layers
        model_results = {}
        for layer in GUS_ANALYSIS_LAYERS:
            model_results[layer] = analyze_layer_gus(layer, gus_layer_acts[layer], gus_labels_np)

        gus_all_results[label] = model_results

        # Build summary table for this model
        gus_summary_rows = []
        for layer, res in model_results.items():
            gus_summary_rows.append({
                "layer": layer, "model": label,
                "n_significant": res["n_significant"],
                "n_active": res["n_active"],
                "probe_auc": res["baseline_auc"],
                "ablation_p": res["ablation_p"],
                "mean_delta_sae": res["mean_delta_sae"],
                "mean_delta_ctrl": res["mean_delta_ctrl"],
                "corr_sae_vs_direct": res["corr_sae_vs_direct"],
                "mean_poly_002": res["mean_poly_002"],
                "effective_dim": res["effective_dim"],
                "mean_angle": res["mean_angle"],
            })
        gus_summary = pd.DataFrame(gus_summary_rows)
        gus_out = MODEL_OUT.parent / label.lower().replace(" ", "_").replace("-", "_")
        gus_out.mkdir(exist_ok=True)
        gus_summary.to_csv(gus_out / "all_layers_summary.csv", index=False)

        print(f"\n{'='*60}")
        print(f"  {label} — ALL-LAYER SUMMARY")
        print(f"{'='*60}")
        print(gus_summary.to_string())

        best_l = gus_summary.loc[gus_summary["probe_auc"].idxmax(), "layer"]
        print(f"\nBest layer by probe AUC: {best_l} (AUC={gus_summary.probe_auc.max():.4f})")

        # Save per-layer results
        for l, res in model_results.items():
            layer_dir = gus_out / f"layer_{l}"
            layer_dir.mkdir(exist_ok=True)
            res["feat_df"].to_csv(layer_dir / "feature_statistics.csv", index=False)
            res["neuron_df"].to_csv(layer_dir / "neuron_importance.csv", index=False)
            res["abl_df"].to_csv(layer_dir / "ablation_results.csv", index=False)

        # Cleanup model from GPU
        del gus_model, gus_tokenizer, gus_layer_acts
        gc.collect()
        if torch.cuda.is_available(): torch.cuda.empty_cache()

    print("\nAll GUS-Net analyses complete.")
else:
    print("GUS dataset not found — skipping GUS-Net analysis.")

In [ ]:
# ── GUS-Net cross-layer visualization ──

if HAS_GUS and len(gus_all_results) > 0:
    fig, axes = plt.subplots(2, 3, figsize=(20, 10))
    colors = {"GUS-Net BERT": "#2ecc71", "GUS-Net GPT-2": "#f39c12"}

    for label, model_res in gus_all_results.items():
        layers = sorted(model_res.keys())
        c = colors.get(label, "#333")

        # (A) Probe AUC
        axes[0, 0].plot(layers, [model_res[l]["baseline_auc"] for l in layers],
                        "o-", color=c, lw=2, label=label)
        # (B) Significant features
        axes[0, 1].plot(layers, [model_res[l]["n_significant"] for l in layers],
                        "o-", color=c, lw=2, label=label)
        # (C) SAE vs direct correlation
        axes[0, 2].plot(layers, [model_res[l]["corr_sae_vs_direct"] for l in layers],
                        "o-", color=c, lw=2, label=label)
        # (D) Polysemanticity
        axes[1, 0].plot(layers, [model_res[l]["mean_poly_002"] for l in layers],
                        "D-", color=c, lw=2, label=label)
        # (E) Ablation delta
        axes[1, 1].plot(layers, [model_res[l]["mean_delta_sae"] for l in layers],
                        "o-", color=c, lw=2, label=f"{label} SAE")
        axes[1, 1].plot(layers, [model_res[l]["mean_delta_ctrl"] for l in layers],
                        "s--", color=c, lw=1, alpha=0.5, label=f"{label} ctrl")
        # (F) Effective dimensionality
        axes[1, 2].plot(layers, [model_res[l]["effective_dim"] for l in layers],
                        "^-", color=c, lw=2, label=label)

    titles = [
        "(A) Probe AUC", "(B) Significant Features", "(C) SAE vs Direct rho",
        "(D) Polysemanticity (tau=0.02)", "(E) Causal Ablation", "(F) Effective Dim"
    ]
    ylabels = ["AUC", "# Significant", "Spearman rho",
               "Features/neuron", "Delta AUC", "Effective dim"]
    for ax, t, yl in zip(axes.flat, titles, ylabels):
        ax.set_title(t, fontweight="bold")
        ax.set_xlabel("Layer")
        ax.set_ylabel(yl)
        ax.legend(fontsize=7)

    axes[1, 1].axhline(0, ls=":", color="black", alpha=0.3)

    plt.suptitle("GUS-Net Models — Cross-Layer Overview", fontsize=14, fontweight="bold", y=1.02)
    plt.tight_layout()
    gus_fig_dir = MODEL_OUT.parent / "gus_net_figures"
    gus_fig_dir.mkdir(exist_ok=True)
    plt.savefig(gus_fig_dir / "gus_net_cross_layer_overview.png", dpi=150, bbox_inches="tight")
    plt.show()
else:
    print("No GUS-Net results to visualize.")

In [ ]:
# ── Base vs GUS-Net comparison: how does fine-tuning change neuron-level bias encoding? ──

if HAS_GUS and len(gus_all_results) > 0 and len(layer_results) > 0:
    fig, axes = plt.subplots(2, 3, figsize=(20, 12))

    base_layers = sorted(layer_results.keys())
    base_auc = [layer_results[l]["baseline_auc"] for l in base_layers]
    base_nsig = [layer_results[l]["n_significant"] for l in base_layers]
    base_poly = [layer_results[l]["mean_poly_002"] for l in base_layers]
    base_delta = [layer_results[l]["mean_delta_sae"] for l in base_layers]
    base_dim = [layer_results[l]["effective_dim"] for l in base_layers]
    base_angle = [layer_results[l]["mean_angle"] for l in base_layers]

    # Plot base model
    base_color = "#e74c3c" if MODEL_FAMILY == "gpt2" else "#3498db"
    base_label = f"Base {MODEL_LABEL}"

    # Determine which GUS-Net model to compare (same family)
    gus_label = f"GUS-Net {'BERT' if MODEL_FAMILY == 'bert' else 'GPT-2'}"
    if gus_label in gus_all_results:
        gus_res = gus_all_results[gus_label]
        gus_layers = sorted(gus_res.keys())
        gus_color = "#2ecc71"

        metrics = [
            ("baseline_auc", "Probe AUC", "(A) Bias Probe AUC"),
            ("n_significant", "# Significant", "(B) Significant Features"),
            ("mean_poly_002", "Features/neuron", "(C) Polysemanticity"),
            ("mean_delta_sae", "Delta AUC", "(D) Causal Ablation"),
            ("effective_dim", "Effective dim", "(E) Superposition"),
            ("mean_angle", "Mean angle (deg)", "(F) Feature Geometry"),
        ]

        for ax, (key, ylabel, title) in zip(axes.flat, metrics):
            ax.plot(base_layers, [layer_results[l][key] for l in base_layers],
                    "o-", color=base_color, lw=2, label=base_label)
            ax.plot(gus_layers, [gus_res[l][key] for l in gus_layers],
                    "s-", color=gus_color, lw=2, label=gus_label)
            ax.set_title(title, fontweight="bold")
            ax.set_xlabel("Layer"); ax.set_ylabel(ylabel)
            ax.legend(fontsize=8)

        axes[1, 0].axhline(0, ls=":", color="black", alpha=0.3)

        plt.suptitle(f"Base {MODEL_LABEL} vs {gus_label} — Effect of Fine-Tuning on Neuron Bias Encoding",
                     fontsize=13, fontweight="bold", y=1.02)
        plt.tight_layout()
        gus_fig_dir = MODEL_OUT.parent / "gus_net_figures"
        gus_fig_dir.mkdir(exist_ok=True)
        plt.savefig(gus_fig_dir / f"base_vs_gusnet_{MODEL_FAMILY}.png", dpi=150, bbox_inches="tight")
        plt.show()

        # Neuron importance correlation between base and fine-tuned (best layers)
        base_best = int(summary_df.loc[summary_df["probe_auc"].idxmax(), "layer"])
        gus_summary_df = pd.DataFrame([{
            "layer": l, "probe_auc": gus_res[l]["baseline_auc"]
        } for l in gus_layers])
        gus_best = int(gus_summary_df.loc[gus_summary_df["probe_auc"].idxmax(), "layer"])

        base_imp = layer_results[base_best]["neuron_importance"]
        gus_imp = gus_res[gus_best]["neuron_importance"]
        rho_bg, p_bg = stats.spearmanr(base_imp, gus_imp)

        fig, ax = plt.subplots(figsize=(7, 7))
        ax.scatter(base_imp, gus_imp, alpha=0.3, s=10, c="#9b59b6")
        ax.set_xlabel(f"Base {MODEL_LABEL} L{base_best} neuron importance")
        ax.set_ylabel(f"{gus_label} L{gus_best} neuron importance")
        ax.set_title(f"Neuron Importance: Base vs Fine-Tuned (rho={rho_bg:.3f}, p={p_bg:.2e})",
                     fontweight="bold")
        plt.tight_layout()
        plt.savefig(gus_fig_dir / f"neuron_importance_base_vs_gusnet_{MODEL_FAMILY}.png",
                    dpi=150, bbox_inches="tight")
        plt.show()

        print(f"Base {MODEL_LABEL} best layer: L{base_best} (AUC={layer_results[base_best]['baseline_auc']:.4f})")
        print(f"{gus_label} best layer: L{gus_best} (AUC={gus_res[gus_best]['baseline_auc']:.4f})")
        print(f"Neuron importance correlation: rho={rho_bg:.4f}, p={p_bg:.2e}")
    else:
        print(f"{gus_label} not found in results. Available: {list(gus_all_results.keys())}")
else:
    print("Need both base model and GUS-Net results for comparison.")

In [ ]:
# ── GUS-Net thesis summary table ──

if HAS_GUS and len(gus_all_results) > 0:
    print("=" * 80)
    print("  GUS-NET THESIS SUMMARY")
    print("=" * 80)

    for label, model_res in gus_all_results.items():
        layers = sorted(model_res.keys())
        aucs = {l: model_res[l]["baseline_auc"] for l in layers}
        best_l = max(aucs, key=aucs.get)
        best = model_res[best_l]

        print(f"\n{'─'*60}")
        print(f"  {label}")
        print(f"{'─'*60}")
        print(f"  Best layer: L{best_l} (probe AUC = {best['baseline_auc']:.4f})")
        print(f"  Significant features (best): {best['n_significant']}")
        print(f"  Polysemanticity (tau=0.02): {best['mean_poly_002']:.1f} features/neuron")
        print(f"  Effective dim (best): {best['effective_dim']:.1f}")
        print(f"  Mean angle (best): {best['mean_angle']:.1f} deg")
        print(f"  SAE vs direct rho: {best['corr_sae_vs_direct']:.4f}")
        print(f"  Ablation p-value: {best['ablation_p']:.4f}")
        print(f"  Ablation delta SAE: {best['mean_delta_sae']:.4f}")
        print(f"  Ablation delta ctrl: {best['mean_delta_ctrl']:.4f}")
        print(f"  Neurons for 90% var: {best['neurons_for_90pct']:.0f}")

    # Compare all 4 models if base model results are available
    if len(layer_results) > 0:
        print(f"\n{'='*80}")
        print("  4-MODEL COMPARISON (Base + GUS-Net)")
        print(f"{'='*80}")

        compare_rows = []
        # Base model
        base_best_l = int(summary_df.loc[summary_df["probe_auc"].idxmax(), "layer"])
        base_best = layer_results[base_best_l]
        compare_rows.append({
            "Model": f"Base {MODEL_LABEL}",
            "Best Layer": base_best_l,
            "Probe AUC": base_best["baseline_auc"],
            "N Sig Features": base_best["n_significant"],
            "Poly (tau=0.02)": base_best["mean_poly_002"],
            "Eff Dim": base_best["effective_dim"],
            "Ablation p": base_best["ablation_p"],
        })
        # GUS-Net models
        for label, model_res in gus_all_results.items():
            layers = sorted(model_res.keys())
            aucs = {l: model_res[l]["baseline_auc"] for l in layers}
            best_l = max(aucs, key=aucs.get)
            best = model_res[best_l]
            compare_rows.append({
                "Model": label,
                "Best Layer": best_l,
                "Probe AUC": best["baseline_auc"],
                "N Sig Features": best["n_significant"],
                "Poly (tau=0.02)": best["mean_poly_002"],
                "Eff Dim": best["effective_dim"],
                "Ablation p": best["ablation_p"],
            })

        compare_df = pd.DataFrame(compare_rows)
        print(compare_df.to_string(index=False))

        # Save
        gus_fig_dir = MODEL_OUT.parent / "gus_net_figures"
        gus_fig_dir.mkdir(exist_ok=True)
        compare_df.to_csv(gus_fig_dir / "base_vs_gusnet_comparison.csv", index=False)
        print(f"\nSaved to {gus_fig_dir / 'base_vs_gusnet_comparison.csv'}")
else:
    print("No GUS-Net results available.")